# 有趣的指标
最长的歌
最短的歌
前奏最长
前奏最短
歌词字数最多
歌词字数最少
歌词重复率最高
歌词重复率最低
歌名最长

In [1]:
import zlib
import re
import thulac
from collections import Counter
from difflib import SequenceMatcher
from typing import Dict, List, Tuple, Any
import math


import pandas as pd
import json

## 歌词重复率

### 文本压缩率

In [2]:
# 使用zlib计算压缩率作为重复率，不严谨
def get_text_repetition_stats(text, min_len=4):
    """
    中英文兼容版：支持汉字统计与英文单词统计
    """
    # 1. 提取有效字符：保留汉字、字母和单词间的空格
    # \u4e00-\u9fa5 汉字 | a-zA-Z 英文
    # 我们先清洗掉标点符号，保留空格以区分英文单词
    clean_text_with_spaces = " ".join(re.findall(r'[\u4e00-\u9fa5]|[a-zA-Z]+', text))
    
    # 2. 计算 total_chars (英文按单词算一个单位，汉字按个算)
    # 但为了保持和之前 total_chars 逻辑一致，我们通常按“字符长度”计算
    # 移除所有多余空格进行基础统计
    clean_text = re.sub(r'\s+', '', clean_text_with_spaces)
    total_chars = len(clean_text)
    
    if total_chars == 0:
        return {"total_chars": 0, "compression_repetition_rate": "0.00%", "repeated_chars_total": 0, "repeat_detail": []}

    # 3. 压缩率 (基于编码后的字节)
    encoded = clean_text.encode('utf-8')
    compression_rate = (1 - len(zlib.compress(encoded, level=9)) / len(encoded)) * 100

    # 4. 重复标记逻辑 (使用 mask 标记被覆盖的字符位)
    covered_mask = [0] * len(clean_text)
    repeated_segments = {}

    # 遍历不同长度的子串
    for length in range(20, min_len - 1, -1):
        for i in range(len(clean_text) - length + 1):
            sub = clean_text[i:i + length]
            # 只有出现 2 次及以上，且不是已被记录的长子串
            if clean_text.count(sub) > 1:
                if not any(sub in existing for existing in repeated_segments):
                    repeated_segments[sub] = clean_text.count(sub)
                
                # 标记位置
                for match in re.finditer(f'(?={re.escape(sub)})', clean_text):
                    start = match.start()
                    for k in range(start, start + length):
                        covered_mask[k] = 1

    rep_chars_total = sum(covered_mask)

    return {
        "total_chars": total_chars,
        "compression_repetition_rate": f"{compression_rate:.2f}%",
        "repeated_chars_total": rep_chars_total,
        # "repeat_detail": sorted(repeated_segments.items(), key=lambda x: x[1], reverse=True)
    }

### 完整版

In [3]:
"""
基于THULAC分词的歌词重复率分析工具
精准计算句子和词语重复率，适合大规模歌词分析
"""
import re
import thulac
from collections import Counter
from difflib import SequenceMatcher
from typing import Dict, List, Tuple, Any
import math

class ThulacLyricAnalyzer:
    """基于THULAC分词的歌词分析器"""
    
    def __init__(self, remove_stopwords: bool = True):
        """
        初始化分析器
        
        Args:
            remove_stopwords: 是否移除停用词
        """
        # 初始化THULAC分词器，使用默认模型
        self.thu = thulac.thulac(seg_only=True)  # seg_only=True只进行分词
        
        # 中文停用词集合（可根据需要扩展）
        self.stopwords = {
            '的', '了', '在', '和', '与', '或', '而', '但', '是', '有',
            '就', '都', '也', '又', '还', '却', '到', '着', '过', '吧',
            '吗', '呢', '啊', '呀', '啦', '哇', '嘛', '哟', '诶', '哦',
            '这', '那', '哪', '你', '我', '他', '她', '它', '们'
        }
        self.remove_stopwords = remove_stopwords
    
    def segment_lyrics(self, lyrics: str) -> List[str]:
        """
        使用THULAC对歌词进行分词
        
        Args:
            lyrics: 原始歌词文本
            
        Returns:
            分词后的词语列表
        """
        # 清洗歌词：移除多余空白、换行符
        lyrics_clean = lyrics.strip()
        lyrics_clean = re.sub(r'\s+', ' ', lyrics_clean)  # 合并空白字符
        
        # 使用THULAC分词
        # thulac返回格式：[(word, pos), (word, pos), ...]
        segmented = self.thu.cut(lyrics_clean, text=True)  # text=True返回字符串
        
        # 分割成词语列表
        words = segmented.split()
        
        # 可选：移除停用词
        if self.remove_stopwords:
            words = [w for w in words if w not in self.stopwords]
        
        return words
    
    def split_sentences(self, lyrics: str) -> List[str]:
        """
        将歌词分割成句子
        
        Args:
            lyrics: 原始歌词文本
            
        Returns:
            句子列表
        """
        # 清洗歌词
        lyrics_clean = lyrics.strip()
        lyrics_clean = re.sub(r'\s+', ' ', lyrics_clean)
        
        # 使用中文标点分割句子
        # 支持：。！？；,!?;、～~…—
        sentences = re.split(r'[。！？；,.!?;、～~…—]', lyrics_clean)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
        
        return sentences
    
    def calculate_sentence_repetition(self, sentences: List[str]) -> Dict[str, float]:
        """
        计算句子级重复率
        
        Args:
            sentences: 句子列表
            
        Returns:
            句子重复率指标
        """
        if len(sentences) <= 1:
            return {
                'exact_repetition_rate': 0.0,
                'similar_repetition_rate': 0.0,
                'repeated_sentence_count': 0,
                'total_sentences': len(sentences)
            }
        
        # 1. 完全相同的句子重复
        sentence_counter = Counter(sentences)
        exact_repeated = sum(count - 1 for count in sentence_counter.values())
        exact_repetition_rate = exact_repeated / len(sentences)
        
        # 2. 相似的句子重复（相似度 > 阈值）
        similarity_threshold = 0.8
        similar_pairs = 0
        total_pairs = 0
        
        for i in range(len(sentences)):
            for j in range(i + 1, len(sentences)):
                similarity = SequenceMatcher(None, sentences[i], sentences[j]).ratio()
                if similarity > similarity_threshold:
                    similar_pairs += 1
                total_pairs += 1
        
        similar_repetition_rate = similar_pairs / total_pairs if total_pairs > 0 else 0
        
        # 3. 计算重复句子的详细信息
        repeated_sentences = {s: c for s, c in sentence_counter.items() if c > 1}
        
        return {
            'exact_repetition_rate': exact_repetition_rate,
            'similar_repetition_rate': similar_repetition_rate,
            'repeated_sentence_count': len(repeated_sentences),
            'total_sentences': len(sentences),
            'sentence_repetition_score': (exact_repetition_rate + similar_repetition_rate) / 2
        }
    
    def calculate_word_repetition(self, words: List[str]) -> Dict[str, float]:
        """
        计算词语级重复率
        
        Args:
            words: 词语列表
            
        Returns:
            词语重复率指标
        """
        if len(words) == 0:
            return {
                'word_repetition_rate': 0.0,
                'repetition_density': 0.0,
                'unique_words': 0,
                'total_words': 0
            }
        
        # 统计词频
        word_counter = Counter(words)
        
        # 1. 重复词语比例
        repeated_words = sum(1 for count in word_counter.values() if count > 1)
        word_repetition_rate = repeated_words / len(word_counter) if len(word_counter) > 0 else 0
        
        # 2. 重复密度（重复次数占总词数的比例）
        total_repeats = sum(count - 1 for count in word_counter.values() if count > 1)
        repetition_density = total_repeats / len(words) if len(words) > 0 else 0
        
        # 3. 计算TF-IDF风格的重复重要性（考虑词频和分布）
        # 这里简化：高频词的权重更高
        word_importance_scores = []
        for word, count in word_counter.items():
            if count > 1:  # 只考虑重复词
                # 重要性 = log(词频) * (词频/总词数)
                importance = math.log(count + 1) * (count / len(words))
                word_importance_scores.append(importance)
        
        avg_importance = sum(word_importance_scores) / len(word_importance_scores) if word_importance_scores else 0
        
        return {
            'word_repetition_rate': word_repetition_rate,
            'repetition_density': repetition_density,
            'word_importance_score': avg_importance,
            'unique_words': len(word_counter),
            'total_words': len(words),
            'vocabulary_richness': len(word_counter) / len(words) if len(words) > 0 else 0
        }
    
    def calculate_ngram_repetition(self, words: List[str], n_values: List[int] = None) -> Dict[str, float]:
        """
        计算N-gram重复率
        
        Args:
            words: 词语列表
            n_values: n-gram的n值列表，默认为[2, 3, 4]
            
        Returns:
            n-gram重复率指标
        """
        if n_values is None:
            n_values = [2, 3, 4]
        
        ngram_metrics = {}
        
        for n in n_values:
            if len(words) >= n:
                # 生成n-gram
                ngrams = []
                for i in range(len(words) - n + 1):
                    ngram = ' '.join(words[i:i+n])
                    ngrams.append(ngram)
                
                # 统计n-gram
                ngram_counter = Counter(ngrams)
                total_ngrams = len(ngrams)
                unique_ngrams = len(ngram_counter)
                
                # 计算重复率
                repetition_rate = (total_ngrams - unique_ngrams) / total_ngrams if total_ngrams > 0 else 0
                ngram_metrics[f'ngram_{n}_repetition'] = repetition_rate
                
                # 找出重复的n-gram
                repeated_ngrams = {ng: cnt for ng, cnt in ngram_counter.items() if cnt > 1}
                ngram_metrics[f'ngram_{n}_repeated_count'] = len(repeated_ngrams)
            else:
                ngram_metrics[f'ngram_{n}_repetition'] = 0.0
                ngram_metrics[f'ngram_{n}_repeated_count'] = 0
        
        # 计算n-gram综合得分（加权平均）
        valid_ngrams = [n for n in n_values if len(words) >= n]
        if valid_ngrams:
            # 给更大的n更高的权重（更大的n代表更长的重复片段）
            weights = {2: 0.2, 3: 0.3, 4: 0.5}  # 权重配置
            total_weight = sum(weights.get(n, 0.3) for n in valid_ngrams)
            weighted_sum = sum(ngram_metrics.get(f'ngram_{n}_repetition', 0) * weights.get(n, 0.3) 
                              for n in valid_ngrams)
            ngram_metrics['ngram_composite_score'] = weighted_sum / total_weight if total_weight > 0 else 0
        else:
            ngram_metrics['ngram_composite_score'] = 0.0
        
        return ngram_metrics
    
    def calculate_comprehensive_repetition(self, 
                                         sentence_metrics: Dict[str, float],
                                         word_metrics: Dict[str, float],
                                         ngram_metrics: Dict[str, float]) -> Dict[str, Any]:
        """
        计算综合重复率
        
        Args:
            sentence_metrics: 句子重复率指标
            word_metrics: 词语重复率指标
            ngram_metrics: n-gram重复率指标
            
        Returns:
            综合重复率指标
        """
        # 定义权重（可调整）
        weights = {
            'sentence_exact': 0.25,      # 完全相同的句子重复
            'sentence_similar': 0.20,     # 相似的句子重复
            'word_repetition': 0.25,      # 词语重复率
            'ngram_composite': 0.30       # n-gram综合重复
        }
        
        # 获取各项分数
        sentence_exact_score = sentence_metrics.get('exact_repetition_rate', 0)
        sentence_similar_score = sentence_metrics.get('similar_repetition_rate', 0)
        word_repetition_score = word_metrics.get('word_repetition_rate', 0)
        ngram_composite_score = ngram_metrics.get('ngram_composite_score', 0)
        
        # 计算加权综合得分
        comprehensive_score = (
            sentence_exact_score * weights['sentence_exact'] +
            sentence_similar_score * weights['sentence_similar'] +
            word_repetition_score * weights['word_repetition'] +
            ngram_composite_score * weights['ngram_composite']
        )
        
        # 计算各维度的贡献度
        contributions = {
            'sentence_exact': sentence_exact_score * weights['sentence_exact'],
            'sentence_similar': sentence_similar_score * weights['sentence_similar'],
            'word_repetition': word_repetition_score * weights['word_repetition'],
            'ngram_composite': ngram_composite_score * weights['ngram_composite']
        }
        
        # 确定主要重复类型
        main_contribution = max(contributions.items(), key=lambda x: x[1])
        
        return {
            'comprehensive_score': comprehensive_score,
            'contributions': contributions,
            'main_repetition_type': main_contribution[0],
            'weighted_scores': {
                'sentence_exact': sentence_exact_score,
                'sentence_similar': sentence_similar_score,
                'word_repetition': word_repetition_score,
                'ngram_composite': ngram_composite_score
            }
        }
    
    def analyze_lyrics(self, lyrics: str) -> Dict[str, Any]:
        """
        完整分析一首歌词
        
        Args:
            lyrics: 歌词文本
            
        Returns:
            完整的分析结果
        """
        # 1. 预处理：分割句子和分词
        sentences = self.split_sentences(lyrics)
        words = self.segment_lyrics(lyrics)
        
        # 2. 计算各项指标
        sentence_metrics = self.calculate_sentence_repetition(sentences)
        word_metrics = self.calculate_word_repetition(words)
        ngram_metrics = self.calculate_ngram_repetition(words)
        
        # 3. 计算综合重复率
        comprehensive = self.calculate_comprehensive_repetition(
            sentence_metrics, word_metrics, ngram_metrics
        )
        
        # 4. 收集详细信息
        sentence_counter = Counter(sentences)
        word_counter = Counter(words)
        
        # 找出重复的句子和词语
        repeated_sentences = {s: c for s, c in sentence_counter.items() if c > 1}
        repeated_words = {w: c for w, c in word_counter.items() if c > 1}
        
        # 按重复次数排序
        top_repeated_sentences = sorted(
            repeated_sentences.items(), 
            key=lambda x: x[1], 
            reverse=True
        )[:5]  # 取前5个
        
        top_repeated_words = sorted(
            repeated_words.items(), 
            key=lambda x: x[1], 
            reverse=True
        )[:10]  # 取前10个
        
        # 5. 组装结果
        result = {
            'basic_statistics': {
                'total_sentences': len(sentences),
                'total_words': len(words),
                'unique_words': word_metrics['unique_words'],
                'vocabulary_richness': word_metrics['vocabulary_richness'],
                'sentence_length_avg': len(words) / len(sentences) if len(sentences) > 0 else 0
            },
            'sentence_repetition': sentence_metrics,
            'word_repetition': word_metrics,
            'ngram_repetition': ngram_metrics,
            'comprehensive_analysis': comprehensive,
            'detailed_findings': {
                'repeated_sentences': [
                    {'sentence': s, 'count': c} for s, c in top_repeated_sentences
                ],
                'repeated_words': [
                    {'word': w, 'count': c} for w, c in top_repeated_words
                ],
                'sentence_examples': sentences[:3] if sentences else []  # 显示前3个句子作为示例
            },
            'interpretation': self.interpret_results(comprehensive['comprehensive_score'])
        }
        
        return result
    
    def interpret_results(self, score: float) -> Dict[str, str]:
        """
        解释分析结果
        
        Args:
            score: 综合重复率得分（0-1）
            
        Returns:
            解释说明
        """
        if score < 0.05:
            level = "极低重复"
            description = "歌词新颖度高，几乎没有重复，属于创新性较强的创作"
            characteristic = "散文诗风格，注重叙事和意象"
        elif score < 0.12:
            level = "低重复"
            description = "重复适度，平衡了新颖性和记忆点"
            characteristic = "典型叙事性流行歌曲，有一定副歌重复"
        elif score < 0.20:
            level = "中等重复"
            description = "有一定重复，符合主流流行歌曲特点"
            characteristic = "传统流行结构，有明显副歌重复"
        elif score < 0.30:
            level = "较高重复"
            description = "重复明显，记忆点突出，易于传唱"
            characteristic = "商业流行歌曲，强调副歌重复和hook"
        elif score < 0.40:
            level = "高重复"
            description = "高度重复，强调节奏和氛围"
            characteristic = "舞曲、电子音乐或说唱风格"
        else:
            level = "极高重复"
            description = "极度重复，通常用于特定音乐效果"
            characteristic = "实验音乐、氛围音乐或儿童歌曲"
        
        return {
            'repetition_level': level,
            'score_interpretation': description,
            'typical_characteristics': characteristic,
            'score_range': f"{score:.3f} (0-1)"
        }
    
    def batch_analyze(self, lyrics_dict: Dict[str, str]) -> Dict[str, Dict[str, Any]]:
        """
        批量分析多首歌词
        
        Args:
            lyrics_dict: 歌词字典，格式为{歌名: 歌词文本}
            
        Returns:
            分析结果字典
        """
        results = {}
        
        for song_name, lyrics in lyrics_dict.items():
            print(f"正在分析: {song_name}")
            try:
                analysis = self.analyze_lyrics(lyrics)
                results[song_name] = analysis
            except Exception as e:
                print(f"分析 {song_name} 时出错: {str(e)}")
                results[song_name] = {'error': str(e)}
        
        return results
    
    def get_summary_report(self, analysis_result: Dict[str, Any]) -> str:
        """
        生成简洁的分析报告
        
        Args:
            analysis_result: 单首歌词的分析结果
            
        Returns:
            文本报告
        """
        basic = analysis_result['basic_statistics']
        sentence_rep = analysis_result['sentence_repetition']
        word_rep = analysis_result['word_repetition']
        comp = analysis_result['comprehensive_analysis']
        interp = analysis_result['interpretation']
        
        report = f"""
歌词重复率分析报告
{'=' * 40}

📊 基本统计:
   句子数: {basic['total_sentences']}
   总词数: {basic['total_words']}
   唯一词数: {basic['unique_words']}
   词汇丰富度: {basic['vocabulary_richness']:.3f}

🎵 重复率分析:
   句子完全重复率: {sentence_rep['exact_repetition_rate']:.3%}
   句子相似重复率: {sentence_rep['similar_repetition_rate']:.3%}
   词语重复率: {word_rep['word_repetition_rate']:.3%}
   N-gram综合重复: {analysis_result['ngram_repetition']['ngram_composite_score']:.3%}

🏆 综合重复率: {comp['comprehensive_score']:.3%}

📈 分析结论:
   重复等级: {interp['repetition_level']}
   特点: {interp['typical_characteristics']}

🔍 主要发现:
   重复句子数: {sentence_rep['repeated_sentence_count']}
   高频词: {', '.join([w['word'] for w in analysis_result['detailed_findings']['repeated_words'][:3]])}
"""
        return report


# 使用示例
if __name__ == "__main__":
    # 示例歌词（五月天《如烟》）
    example_lyrics = """我坐在床前。望着窗外回忆满天。生命是华丽错觉。时间是贼偷走一切。七岁的那一年。抓住那只蝉。以为能抓住夏天。十七岁的那年。吻过他的脸。就以为和他能永远。有没有那么一种永远。永远不改变。拥抱过的美丽。都再也不破碎。让险峻岁月不能。在脸上撒野。让生离和死别都遥远。有谁能听见。我坐在床前。转过头看谁在沉睡。那一张苍老的脸。好像是我紧闭双眼。曾经是爱我的。和我深爱的。都围绕在我身边。带不走的那些。遗憾和眷恋。就化成最后一滴泪。有没有那么一滴眼泪。能洗掉后悔。化成大雨降落在。回不去的街。再给我一次机会。将故事改写。还欠了他一生的。一句抱歉。有没有那么一个世界。永远不天黑。星星太阳万物都。听我的指挥。月亮不忙着圆缺。春天不走远。树梢紧紧拥抱着树叶。有谁能听见。耳际眼前此生重演。是我来自漆黑。而又回归漆黑。人间瞬间天地之间。下次我又是谁。有没有那么一朵玫瑰。永远不凋谢。永远骄傲和完美。永远不妥协。为何人生最后会像。一张纸屑。还不如一片花瓣。曾经鲜艳。有没有那么一张书签。停止那一天。最单纯的笑脸和。最美那一年。书包里面装满了。蛋糕和汽水。双眼只有无猜和无邪。让我们无法无天。有没有那么一首诗篇。找不到句点。青春永远定居在。我们的岁月。男孩和女孩都有。吉他和舞鞋。笑忘人间的苦痛。只有甜美。有没有那么一个明天。重头活一遍。让我再次感受。曾挥霍的昨天。无论生存或生活。我都不浪费。不让故事这么的后悔。有谁能听见。我不要告别。我坐在床前。看着指尖已经如烟。"""
    
    # 创建分析器
    analyzer = ThulacLyricAnalyzer(remove_stopwords=True)
    
    # 分析单首歌词
    print("正在分析歌词...")
    result = analyzer.analyze_lyrics(example_lyrics)
    
    # 输出报告
    print(analyzer.get_summary_report(result))
    
    # 批量分析示例
    print("\n" + "="*50)
    print("批量分析示例")
    print("="*50)
    
    # 创建多首歌词的字典
    lyrics_collection = {
        "如烟": example_lyrics,
        "简单歌词示例": "我爱你。我爱你。你爱我。你爱我。我们相爱。永远在一起。",
        "重复较多的歌词": "夜空中最亮的星。夜空中最亮的星。能否听清。能否听清。那仰望的人。那仰望的人。心底的孤独和叹息。心底的孤独和叹息。"
    }
    
    batch_results = analyzer.batch_analyze(lyrics_collection)
    
    for song_name, analysis in batch_results.items():
        if 'error' not in analysis:
            score = analysis['comprehensive_analysis']['comprehensive_score']
            level = analysis['interpretation']['repetition_level']
            print(f"{song_name}: 重复率={score:.3%} ({level})")

Model loaded succeed
正在分析歌词...

歌词重复率分析报告

📊 基本统计:
   句子数: 84
   总词数: 369
   唯一词数: 187
   词汇丰富度: 0.507

🎵 重复率分析:
   句子完全重复率: 4.762%
   句子相似重复率: 0.201%
   词语重复率: 21.925%
   N-gram综合重复: 7.543%

🏆 综合重复率: 8.975%

📈 分析结论:
   重复等级: 低重复
   特点: 典型叙事性流行歌曲，有一定副歌重复

🔍 主要发现:
   重复句子数: 2
   高频词: 。, 一, 不


批量分析示例
正在分析: 如烟
正在分析: 简单歌词示例
正在分析: 重复较多的歌词
如烟: 重复率=8.975% (低重复)
简单歌词示例: 重复率=28.732% (较高重复)
重复较多的歌词: 重复率=50.804% (极高重复)


### 简化版

In [ ]:
"""
简化版歌词重复率分析器
针对单首歌词，仅计算词重复率、句子重复率和综合重复率
SimpleLyricAnalyzer 采用双维度加权综合的方法计算歌词重复率。首先，使用THULAC对歌词进行专业分词并过滤虚词，按标点分割句子。词语重复率通过统计重复词语出现次数占总词数的比例计算；句子重复率基于完全相同的句子数量比例计算。最后，按照预设权重（默认句子占80%，词语占20%）加权平均得出综合重复率。这种方法平衡了结构重复和内容重复，能准确反映歌词的重复程度，适合快速批量分析单段歌词。
"""


class SimpleLyricAnalyzer:
    """简化版歌词重复率分析器"""

    def __init__(self,
                 sentence_similarity_threshold: float = 0.8,
                 word_weight: float = 0.2,
                 sentence_weight: float = 0.8):
        """
        初始化分析器
        
        Args:
            sentence_similarity_threshold: 句子相似度阈值（默认0.8）未使用
            word_weight: 词语重复率权重（默认0.4）
            sentence_weight: 句子重复率权重（默认0.6）
        """
        self.sentence_threshold = sentence_similarity_threshold
        self.word_weight = word_weight
        self.sentence_weight = sentence_weight

        # 初始化THULAC分词器
        self.thu = thulac.thulac(seg_only=True)

        # 常见虚词（用于分词后过滤）
        # self.stopwords = {
        #     '的', '了', '在', '和', '与', '或', '而', '但', '是', '有', '就', '都', '也',
        #     '又', '还', '却', '到', '着', '过', '吧', '吗', '呢', '啊', '呀', '啦', '哇',
        #     '嘛', '哟', '诶', '哦', '这', '那', '哪', '你', '我', '他', '她', '它', '们'
        # }

    def split_sentences(self, lyrics: str) -> list:
        """分割歌词为句子"""
        lyrics_clean = lyrics.strip()
        lyrics_clean = re.sub(r'\s+', ' ', lyrics_clean)

        # 使用中文标点分割
        sentences = re.split(r'[。！？；,.!?;、～~…—]', lyrics_clean)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 0]

        return sentences

    def segment_words(self, lyrics: str) -> list:
        """使用THULAC分词"""
        lyrics_clean = lyrics.strip()
        lyrics_clean = re.sub(r'\s+', ' ', lyrics_clean)

        # THULAC分词
        segmented = self.thu.cut(lyrics_clean, text=True)
        words = segmented.split()

        # 过滤常见虚词
        # words = [w for w in words if w not in self.stopwords]

        return words

    def calculate_sentence_repetition_rate(self, sentences: list) -> float:
        """计算句子重复率"""
        if len(sentences) <= 1:
            return 0.0

        # 统计完全相同的句子
        sentence_counter = Counter(sentences)
        total_repeated = sum(count - 1 for count in sentence_counter.values())

        # 句子重复率 = 重复句子数 / 总句子数
        sentence_repetition = total_repeated / len(sentences)

        return sentence_repetition

    def calculate_word_repetition_rate(self, words: list) -> float:
        """计算词语重复率"""
        if len(words) == 0:
            return 0.0

        # 统计词频
        word_counter = Counter(words)

        # 词语重复率 = 重复词数 / 总词数
        repeated_words = sum(count - 1 for count in word_counter.values()
                             if count > 1)
        word_repetition = repeated_words / len(words)

        return word_repetition

    def analyze(self, lyrics: str) -> dict:
        """
        分析歌词重复率
        
        Args:
            lyrics: 歌词文本
            
        Returns:
            包含重复率结果的字典
        """
        # 1. 分割句子
        sentences = self.split_sentences(lyrics)

        # 2. 分词
        words = self.segment_words(lyrics)

        # 3. 计算句子重复率
        sentence_rate = self.calculate_sentence_repetition_rate(sentences)

        # 4. 计算词语重复率
        word_rate = self.calculate_word_repetition_rate(words)

        # 5. 计算综合重复率（加权平均）
        comprehensive_rate = (sentence_rate * self.sentence_weight +
                              word_rate * self.word_weight)

        return {
            'word_repetition_rate': word_rate,
            'sentence_repetition_rate': sentence_rate,
            'comprehensive_repetition_rate': comprehensive_rate,
            'sentence_count': len(sentences),
            'word_count': len(words),
            'unique_word_count': len(set(words))
        }


# 使用示例
if __name__ == "__main__":
    # 示例歌词
    lyrics = """我坐在床前。望着窗外回忆满天。生命是华丽错觉。时间是贼偷走一切。七岁的那一年。抓住那只蝉。以为能抓住夏天。十七岁的那年。吻过他的脸。就以为和他能永远。有没有那么一种永远。永远不改变。拥抱过的美丽。都再也不破碎。让险峻岁月不能。在脸上撒野。让生离和死别都遥远。有谁能听见。我坐在床前。转过头看谁在沉睡。那一张苍老的脸。好像是我紧闭双眼。曾经是爱我的。和我深爱的。都围绕在我身边。带不走的那些。遗憾和眷恋。就化成最后一滴泪。有没有那么一滴眼泪。能洗掉后悔。化成大雨降落在。回不去的街。再给我一次机会。将故事改写。还欠了他一生的。一句抱歉。有没有那么一个世界。永远不天黑。星星太阳万物都。听我的指挥。月亮不忙着圆缺。春天不走远。树梢紧紧拥抱着树叶。有谁能听见。耳际眼前此生重演。是我来自漆黑。而又回归漆黑。人间瞬间天地之间。下次我又是谁。有没有那么一朵玫瑰。永远不凋谢。永远骄傲和完美。永远不妥协。为何人生最后会像。一张纸屑。还不如一片花瓣。曾经鲜艳。有没有那么一张书签。停止那一天。最单纯的笑脸和。最美那一年。书包里面装满了。蛋糕和汽水。双眼只有无猜和无邪。让我们无法无天。有没有那么一首诗篇。找不到句点。青春永远定居在。我们的岁月。男孩和女孩都有。吉他和舞鞋。笑忘人间的苦痛。只有甜美。有没有那么一个明天。重头活一遍。让我再次感受。曾挥霍的昨天。无论生存或生活。我都不浪费。不让故事这么的后悔。有谁能听见。我不要告别。我坐在床前。看着指尖已经如烟。"""

    # 创建分析器
    analyzer = SimpleLyricAnalyzer()

    # 分析歌词
    result = analyzer.analyze(lyrics)

    # 输出结果
    print(f"词语重复率: {result['word_repetition_rate']:.2%}")
    print(f"句子重复率: {result['sentence_repetition_rate']:.2%}")
    print(f"综合重复率: {result['comprehensive_repetition_rate']:.2%}")
    print(f"\n统计信息:")
    print(f"  句子数: {result['sentence_count']}")
    print(f"  总词数: {result['word_count']}")
    print(f"  唯一词数: {result['unique_word_count']}")

Model loaded succeed
词语重复率: 57.14%
句子重复率: 4.76%
综合重复率: 25.71%

统计信息:
  句子数: 84
  总词数: 483
  唯一词数: 207


## 前奏时长

In [5]:
# start_time为分秒格式，如00:15.71，取.前部分，并转换为秒，作为前奏时长
def parse_intro_duration(start_time_series):
    """
    将start_time(格式如00:15.71)转换为前奏时长(秒)
    
    参数:
        start_time_series: pandas Series, 包含start_time字符串
    
    返回:
        pandas Series, 前奏时长(秒)
    """
    start_time_parts = start_time_series.astype(str).str.split('.', n=1, expand=True)[0]
    start_time_split = start_time_parts.str.split(':', n=1, expand=True)
    
    minutes = pd.to_numeric(start_time_split[0], errors='coerce').fillna(0).astype(int)
    seconds = pd.to_numeric(start_time_split[1], errors='coerce').fillna(0).astype(int)
    
    return minutes * 60 + seconds

## 歌曲时长

In [6]:
# 歌曲时长，将duration转为xx分xx秒文本格式，并保留duration字段
def format_duration(duration_seconds):
    minutes = duration_seconds // 60
    seconds = duration_seconds % 60
    return f"{minutes}分{seconds}秒"

## 歌名字数

In [7]:
def count_song_name(name):
    if not name:
        return 0
    # 1. 匹配所有英文单词 (Now, You, See, Me)
    english_words = re.findall(r'[a-zA-Z0-9]+', name)
    # 2. 匹配所有中文字符 (去除空格后的非英文字符)
    # 先去掉空格，再去掉英文和数字，剩下的就是中文字符或符号
    remaining = re.sub(r'\s+|[a-zA-Z0-9]+', '', name)
    
    # 单词数 + 中文字符数
    return len(english_words) + len(remaining)

## 计算

In [8]:
def analyze_songs_metrics(path_prefix):
    songs_df = pd.read_csv(path_prefix + "cleared_song_data.csv")
    lyrics_df = pd.read_json(path_prefix + "cleared_lyric_data.json",
                             convert_dates=False)
    # 删除lyrics_df中没有歌词的行
    lyrics_df = lyrics_df[lyrics_df['has_lyric'] == 1]
    # lyrics_df 连接 songs_df， 以song_id为键，以lyrics_df为主，保留所有歌词数据，重复列只保留lyrics_df的
    df = pd.merge(lyrics_df,
                  songs_df,
                  on='song_id',
                  how='left',
                  suffixes=('', '_song'))
    df['intro_duration_seconds'] = parse_intro_duration(df['start_time'])
    # 歌曲时长
    df['duration_text'] = df['duration'].apply(format_duration)
    # 重复率，使用SimpleLyricAnalyzer()
    analyzer = SimpleLyricAnalyzer(word_weight=0.2, sentence_weight=0.8)
    result = df['lyrics_text'].apply(analyzer.analyze)
    result_df = pd.DataFrame(result.tolist())
    df = pd.concat([df, result_df], axis=1)

    # 歌词字数，使用get_text_repetition_stats
    metrics = df['lyrics_text'].apply(get_text_repetition_stats)
    metrics_df = pd.DataFrame(metrics.tolist())
    df = pd.concat([df, metrics_df], axis=1)
    # 歌名清洗，去掉括号及其中内容，去掉引号，仅计算汉字数和单词数
    df['clean_song_name'] = (df['song_name'].fillna('').str.replace(
        r'（[^）]*）', '',
        regex=True).str.replace(r'【[^】]*】', '', regex=True).str.replace(
            r'\([^)]*\)', '', regex=True).str.replace(r'[\'"“”]',
                                                      '',
                                                      regex=True).str.strip())
    # 歌名长度。不含空格的字符长度，英文单词按一个单位算
    df['song_name_length'] = df['clean_song_name'].apply(count_song_name)
    return df

In [37]:
def get_extreme_metrics(df):
    # 1. 预处理：数值化重复率以便比较
    comp_numeric = df['comprehensive_repetition_rate'].apply(lambda x: round(x,2))

    def _get_extrema(col_name, use_series=None, text_col=None):
        """
        内部辅助函数：获取极值对应的歌曲名拼接字符串、数值、以及对应的文本展示字段
        """
        series = use_series if use_series is not None else df[col_name]
        if series.empty or series.isna().all():
            return "无", 0, "无", 0, "0", "0"

        v_max, v_min = series.max(), series.min()

        # 提取所有并列的歌名
        names_max = "，".join(df.loc[series == v_max,
                                    'song_name'].unique().astype(str))
        names_min = "，".join(df.loc[series == v_min,
                                    'song_name'].unique().astype(str))

        # 提取对应的展示文本 (取并列组中的第一个)
        t_max = str(v_max)
        t_min = str(v_min)
        if text_col and text_col in df.columns:
            t_max = df.loc[series == v_max, text_col].iloc[0]
            t_min = df.loc[series == v_min, text_col].iloc[0]

        return names_max, v_max, names_min, v_min, t_max, t_min

    # 2. 计算各个维度的极值
    # 时长：传入 duration_text 字段用于展示
    d_names_max, d_max, d_names_min, d_min, d_text_max, d_text_min = _get_extrema(
        'duration', text_col='duration_text')

    # 前奏
    i_names_max, i_max, i_names_min, i_min, _, _ = _get_extrema(
        'intro_duration_seconds')

    # 字数
    c_names_max, c_max, c_names_min, c_min, _, _ = _get_extrema('total_chars')

    # 重复率
    r_names_max, r_max, r_names_min, r_min, _, _ = _get_extrema(
        None, use_series=comp_numeric)

    # 歌名长度
    n_names_max, n_max, _, _, _, _ = _get_extrema('song_name_length')

    # 3. 组装结果字典
    metrics = {
        # 时长统计 - 使用文本化字段
        "最长的歌": d_names_max,
        "最长的歌时长": d_text_max,
        "longest_song_number": int(d_max),
        "最短的歌": d_names_min,
        "最短的歌时长": d_text_min,
        "shortest_song_number": int(d_min),

        # 前奏统计
        "前奏最长": i_names_max,
        "前奏最长时长": f"{i_max}秒",
        "longest_intro_number": int(i_max),
        "前奏最短": i_names_min,
        "前奏最短时长": f"{i_min}秒",
        "shortest_intro_number": int(i_min),

        # 歌词统计
        "歌词字数最多": c_names_max,
        "歌词字数最多字数": f"{c_max}字",
        "most_lyrics_number": int(c_max),
        "歌词字数最少": c_names_min,
        "歌词字数最少字数": f"{c_min}字",
        "least_lyrics_number": int(c_min),

        # 重复率统计
        "歌词重复率最高": r_names_max,
        "歌词重复率最高重复率": f"{int(r_max*100)}%",
        "歌词重复率最低": r_names_min,
        "歌词重复率最低重复率": f"{int(r_min*100)}%",

        # 歌名统计
        "歌名最长": n_names_max,
        "歌名最长字数": f"{n_max}字",
        "longest_song_name_number": int(n_max)
    }

    metrics_vue_data = {
        "singer": df['artist_name'].iloc[0],
        "song": {
            "title":
            "歌曲时长（秒）",
            "category": [
                f"最长：{d_names_max} - {d_text_max}",
                f"最短：{d_names_min} - {d_text_min}"
            ],
            "number": [int(d_max), int(d_min)]
        },
        "intro": {
            "title":
            "前奏时长（秒）",
            "category":
            [f"最长：{i_names_max} - {i_max}秒", f"最短：{i_names_min} - {i_min}秒"],
            "number": [int(i_max), int(i_min)]
        },
        "lyrics": {
            "title":
            "歌词字数",
            "category":
            [f"最多：{c_names_max} - {c_max}字", f"最少：{c_names_min} - {c_min}字"],
            "number": [int(c_max), int(c_min)]
        },
        "repetition": {
            "title":
            "歌词重复率（%）",
            "category":
            [f"最高：{r_names_max} - {int(r_max*100)}%", f"最低：{r_names_min} - {int(r_min*100)}%"],
            "number": [int(r_max*100), int(r_min*100)]
        },
        # "song_name": {
        #     "category": [f"最长：{n_names_max} - {n_max}字"],
        #     "number": [int(n_max)]
        # }
    }
    return metrics, metrics_vue_data
    # return metrics_vue_data

# main

In [38]:
file_path_prefix = "data/jaychou/"

In [39]:
file_path_prefix = "data/jaychou/"
df = analyze_songs_metrics(file_path_prefix)
jay_res = get_extreme_metrics(df)
jay_res

Model loaded succeed


({'最长的歌': '以父之名',
  '最长的歌时长': '5分42秒',
  'longest_song_number': 342,
  '最短的歌': '阳明山',
  '最短的歌时长': '2分32秒',
  'shortest_song_number': 152,
  '前奏最长': '半兽人',
  '前奏最长时长': '63秒',
  'longest_intro_number': 63,
  '前奏最短': '公主病，免费教学录影带',
  '前奏最短时长': '0秒',
  'shortest_intro_number': 0,
  '歌词字数最多': '以父之名',
  '歌词字数最多字数': '1226字',
  'most_lyrics_number': 1226,
  '歌词字数最少': '蒲公英的约定',
  '歌词字数最少字数': '227字',
  'least_lyrics_number': 227,
  '歌词重复率最高': '可爱女人',
  '歌词重复率最高重复率': '73%',
  '歌词重复率最低': '前世情人',
  '歌词重复率最低重复率': '19%',
  '歌名最长': '给我一首歌的时间',
  '歌名最长字数': '8字',
  'longest_song_name_number': 8},
 {'singer': '周杰伦',
  'song': {'title': '歌曲时长（秒）',
   'category': ['最长：以父之名 - 5分42秒', '最短：阳明山 - 2分32秒'],
   'number': [342, 152]},
  'intro': {'title': '前奏时长（秒）',
   'category': ['最长：半兽人 - 63秒', '最短：公主病，免费教学录影带 - 0秒'],
   'number': [63, 0]},
  'lyrics': {'title': '歌词字数',
   'category': ['最多：以父之名 - 1226字', '最少：蒲公英的约定 - 227字'],
   'number': [1226, 227]},
  'repetition': {'title': '歌词重复率（%）',
   'category': ['最高：可爱

In [12]:
df

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,song_mid,song_name_song,...,sentence_repetition_rate,comprehensive_repetition_rate,sentence_count,word_count,unique_word_count,total_chars,compression_repetition_rate,repeated_chars_total,clean_song_name,song_name_length
0,97773,晴天,00:29.26,1,周杰伦,周杰伦,周杰伦,故事的小黄花。从出生那年就飘着。童年的荡秋千。随记忆一直晃到现在。Re，So，So，Si，D...,0039MnYb0qxYhV,晴天,...,0.400000,0.470993,50,453,111,511,66.55%,454,晴天,2
1,102065756,七里香,00:27.74,1,方文山,周杰伦,钟兴民,窗外的麻雀在电线杆上多嘴。你说这一句很有夏天的感觉。手中的铅笔在纸上来来回回。我用几行字形容...,004Z8Ihr0JIu5s,七里香,...,0.382353,0.436239,34,336,117,406,57.22%,252,七里香,3
2,449205,稻香,00:30.94,1,周杰伦,周杰伦,黄雨勋,对这个世界如果你有太多的抱怨。跌倒了就不敢继续往前走。为什么人要这么的脆弱堕落。请你打开电视...,003aAYrm3GE0Ac,稻香,...,0.272727,0.335500,44,358,148,443,52.37%,215,稻香,2
3,410316,青花瓷,00:21.97,1,方文山,周杰伦,钟兴民,素胚勾勒出青花笔锋浓转淡。瓶身描绘的牡丹一如你初妆。冉冉檀香透过窗心事我了然。宣纸上走笔至此...,002qU5aY3Qu24y,青花瓷,...,0.425000,0.465697,40,323,120,418,56.78%,246,青花瓷,3
4,449198,花海,00:27.24,1,古小力/黄淩嘉,周杰伦,黄雨勋,静止了，所有的花开。遥远了，清晰了爱。天郁闷，爱却很喜欢。那时候我不懂这叫爱。你喜欢，站在那...,003cI52o4daJJL,花海,...,0.476190,0.519662,42,248,76,260,58.59%,228,花海,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,268352018,Mojito,00:16.80,1,黄俊郎,周杰伦,黄雨勋,麻烦给我的爱人来一杯Mojito。我喜欢阅读她微醺时的眼眸。而我的咖啡，糖不用太多。这世界已...,001glaI72k8BQX,Mojito,...,0.411765,0.454412,34,320,120,405,53.33%,287,Mojito,1
157,213922043,不爱我就拉倒,00:11.11,1,周杰伦/宋健彰,周杰伦,Hans陈思翰,寒流来了，刚好。刚好可以，把你手放外套。把安全帽戴好。不让你在，爱情路上跌倒。加速狂飙，你说...,0031TAKo0095np,不爱我就拉倒,...,0.530612,0.590239,49,327,56,347,71.95%,337,不爱我就拉倒,6
158,212877900,等你下课 (with 杨瑞代),00:18.77,1,周杰伦,周杰伦,黄雨勋,我租了一间公寓。为了想与你不期而遇。高中三年，我为什么。为什么不好好读书。没考上跟你一样的大...,001J5QJL1pRQYB,等你下课 (with 杨瑞代),...,0.243902,0.321612,41,302,111,322,50.62%,182,等你下课,4
159,237773700,说好不哭 (with 五月天阿信),00:26.51,1,方文山,周杰伦,黄雨勋,没有了联络，后来的生活。我都是听别人说。说你怎么了，说你怎么过。放不下的人是我。人多的时候，...,001qvvgF38HVc4,说好不哭 (with 五月天阿信),...,0.300000,0.382756,30,283,81,294,55.56%,210,说好不哭,4


In [40]:
file_path_prefix = "data/mayday/"
df = analyze_songs_metrics(file_path_prefix)
mayday_res = get_extreme_metrics(df)
mayday_res

Model loaded succeed


({'最长的歌': '温柔 (还你自由版)',
  '最长的歌时长': '7分6秒',
  'longest_song_number': 426,
  '最短的歌': '为什么（今日的爱情）',
  '最短的歌时长': '1分53秒',
  'shortest_song_number': 113,
  '前奏最长': '孙悟空',
  '前奏最长时长': '63秒',
  'longest_intro_number': 63,
  '前奏最短': '派对动物，夜访吸血鬼，雨眠',
  '前奏最短时长': '0秒',
  'shortest_intro_number': 0,
  '歌词字数最多': '干杯',
  '歌词字数最多字数': '700字',
  'most_lyrics_number': 700,
  '歌词字数最少': '后青春期的诗，金多虾',
  '歌词字数最少字数': '175字',
  'least_lyrics_number': 175,
  '歌词重复率最高': '天使',
  '歌词重复率最高重复率': '71%',
  '歌词重复率最低': '后青春期的诗',
  '歌词重复率最低重复率': '7%',
  '歌名最长': '有些事现在不做 一辈子都不会做了',
  '歌名最长字数': '15字',
  'longest_song_name_number': 15},
 {'singer': '五月天',
  'song': {'title': '歌曲时长（秒）',
   'category': ['最长：温柔 (还你自由版) - 7分6秒', '最短：为什么（今日的爱情） - 1分53秒'],
   'number': [426, 113]},
  'intro': {'title': '前奏时长（秒）',
   'category': ['最长：孙悟空 - 63秒', '最短：派对动物，夜访吸血鬼，雨眠 - 0秒'],
   'number': [63, 0]},
  'lyrics': {'title': '歌词字数',
   'category': ['最多：干杯 - 700字', '最少：后青春期的诗，金多虾 - 175字'],
   'number': [700, 175]},
  'repetition': {'title

In [43]:
df[df['song_name']=='后青春期的诗']

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,song_mid,song_name_song,...,sentence_repetition_rate,comprehensive_repetition_rate,sentence_count,word_count,unique_word_count,total_chars,compression_repetition_rate,repeated_chars_total,clean_song_name,song_name_length
45,447823,后青春期的诗,00:01.84,1,阿信,阿信,五月天,当烟雾随晨光飘散。枕畔的湖已风干。期待已退化成等待。而我告别了突然。当泪痕勾勒着遗憾。回忆夸...,003ZXEKi1eQPuX,后青春期的诗,...,0.0,0.070748,26,147,95,175,25.14%,0,后青春期的诗,6


In [42]:
res = {"jay": jay_res[1], "mayday": mayday_res[1]}
# 保存为json文件
with open('data/extreme_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(res, f, ensure_ascii=False, indent=4)

In [29]:
531/3*4

708.0